# Dramatiq

Dramatiq API 比 Celery 更轻，适合 Redis/RabbitMQ 上的后台任务。常用能力包括 actor、重试、结果存储和 pipeline。

```zsh
dramatiq python_notes.task_queues.dramatiq.tasks
python -m python_notes.task_queues.dramatiq.client
```


In [ ]:
import os
from typing import Final

import dramatiq
from dramatiq.brokers.redis import RedisBroker
from dramatiq.results import Results
from dramatiq.results.backends import RedisBackend

DEFAULT_REDIS_URL: Final[str] = 'redis://localhost:6379/0'
REDIS_URL: Final[str] = os.getenv('TASK_QUEUE_REDIS_URL', DEFAULT_REDIS_URL)

result_backend = RedisBackend(url=REDIS_URL)
redis_broker = RedisBroker(url=REDIS_URL)
redis_broker.add_middleware(Results(backend=result_backend))
dramatiq.set_broker(redis_broker)


@dramatiq.actor(actor_name='notebook_dramatiq_add', max_retries=3, min_backoff=1_000, store_results=True)
def dramatiq_add(x: int, y: int) -> int:
    return x + y


In [ ]:
import dramatiq

from python_notes.task_queues.dramatiq.tasks import add, multiply

message = add.send(3, 5)
print('Message ID:', message.message_id)
print('Result:', message.get_result(block=True, timeout=10_000))

workflow = dramatiq.pipeline([add.message(2, 3), multiply.message(10)]).run()
print('Pipeline Result:', workflow.get_result(block=True, timeout=10_000))
